In [1]:
import numpy as np
import iteround as it
import os
import sys
path = os.path.abspath("")
splitted = path.split(os.sep)
user_independent = os.path.join(splitted[0]+os.sep, splitted[1], splitted[2], splitted[3])
src_path = os.path.join(user_independent, 'GitHub', 'Photoswitching')
sys.path.append(src_path)

import src.antibunching as ab
import src.blinking as bl
import src.distributions as dist
import src.emissions as em
import src.fcs as fc
import src.figure as fi
import src.fluorophores as fl
import src.formulas as fo
import src.miscellaneous as mi
import src.network as net
import src.simulation as si
import src.statistics as st
import src.tcspc as tc
import src.transitions as tr

import cupy as cp

c:\Users\ebert\miniconda3\envs\markovmodels\lib\site-packages\pycorrelate\pycorrelate.py:118: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  def ucorrelate(t, u, maxlag=None):


In [2]:
def round_retain_sum(array, topline):
    values = array

    keep_track = cp.arange(0, array.size)
    orig_sum = cp.round(topline, decimals=0)
    new_round = cp.round(values, decimals=0)
    local_sum = cp.round(cp.sum(new_round), decimals=0)

    while local_sum != orig_sum:
        diff = cp.round(orig_sum - local_sum, decimals=0)
        if diff < 0.:
            increment = -1
            reverse = False
        else:
            increment = 1
            reverse = True
        tweaks = cp.abs(diff).astype(int)
        new_round, keep_track = sort_by_diff(array, new_round, keep_track, reverse)
        iterations = cp.arange(0, cp.amin(cp.append(tweaks, array.size)))
        for ith in iterations:
            new_round[ith] += increment
        local_sum = cp.round(cp.sum(new_round), decimals=0)

    reverse_sorting = np.argsort(keep_track)
    rounded_array = new_round[reverse_sorting]
    return rounded_array

def sort_by_diff(array, rounded_array, keep_track, reverse):
    diff = array - rounded_array
    if reverse:
        sorted = cp.argsort(diff)[::-1]
    else:
        sorted = cp.argsort(diff)
    return rounded_array[sorted], keep_track[sorted]


In [3]:
fluorophore_1 = fl.Fluorophore(name='cy5', position=[0, 4], parameter_set='set 1')
fluorophore_system = fl.FluorophoreSystem([fluorophore_1])

In [4]:
transitions = fluorophore_system.load_transitions()

In [5]:
transition_set = tr.TransitionSet(transitions, fluorophore_system)
transition_set.finalize()

In [6]:
prediction = st.Prediction(transition_set)

In [7]:
from torch import tensor, cuda, repeat_interleave

In [8]:
print(cuda.device_count())
print(cuda.current_device())
print(cp.cuda.runtime.getDeviceCount())
print(cp.cuda.runtime.getDevice())

1
0
1
0


In [11]:
def fill_individual_transitions(prediction, transitions, size, seed):
    """
    Constructs an approximation of simulated stochastic data based on a predicted stationary distribution of 
    a Markov chain. 
    The transitions are ordered via a topological sort and processed accordingly. Successor transitions are 
    placed behind their predecessors. The topological sort is possible via a temporary conversion of the graph
    to a directed acyclic graph (DAG).

    Parameters
    ----------
    prediction : src.statistics.Prediction
        Container of lifetimes, state and transition occurrences obtained by computation-associated attributes and
        methods.
    size : int
        Maximum number of simulation steps. Due to rounding, actual size might vary.
    seed : None, int, BitGenerator, Generator
        A seed to initialize the BitGenerator.

    Returns
    -------
    time_series : 1-D array_like
        The simulated time points. At index i, they correspond to transition_series[i - 1].
    transition_series : 1-D array_like
        The simulated transitions. At index i, they correspond to time_series[i + 1].
    """
    transition_occurrences = prediction.stationary_distribution_transitions * size
    transition_occurrences = transition_occurrences.astype(cp.int64)
    maximum_transition_index = np.argmax(transition_occurrences)
    
    starting_transition = transitions.transition_df.index[maximum_transition_index]
    
    graph = net.construct_graph_transitions(transitions.transition_df, numerical=True)
    graph_suited, _ = net.check_graph_suitable(G=graph, starting_node=starting_transition)
    if not graph_suited:
        raise ValueError('graph is not suited for the algorithm. Check for loops that do not contain the most occurring state.')
    transition_order = net.determine_node_order(G=graph, starting_node=starting_transition)
    
    all_rates = cp.array(transitions.transition_df['rate'])
    rng = cp.random.default_rng(seed)    
    transition_series = cp.full(transition_occurrences[maximum_transition_index], starting_transition)
    for i, transition in enumerate(transition_order):
        transition_indices = cp.where(transition_series == transition)[0]
        occurrences = transition_indices.size
        cp.random.shuffle(transition_indices)
        follow_up_transitions = cp.array(list(graph.successors(transition)))
        rates = all_rates[follow_up_transitions]
        repeats = rates * occurrences / rates.sum()
        rounded_repeats = round_retain_sum(repeats, topline=occurrences)
        rounded_repeats = cp.array(rounded_repeats, dtype=cp.int64)
        if starting_transition in follow_up_transitions:
            if follow_up_transitions.size == 1:
                continue
            # no np.delete in cupy or pytorch
            indices_not_starting_transition = cp.where(follow_up_transitions != starting_transition)[0]
            follow_up_transitions = follow_up_transitions[indices_not_starting_transition]
            number_of_deletions = rounded_repeats.sum() - rounded_repeats[indices_not_starting_transition].sum()
            rounded_repeats = rounded_repeats[indices_not_starting_transition]
            transition_indices = transition_indices[:-number_of_deletions]
        
        # no np.repeat in cupy
        # but why
        #follow_up_transitions = follow_up_transitions.get()
        follow_up_transitions_torch = tensor(follow_up_transitions, device='cuda')
        #rounded_repeats = rounded_repeats.get()
        rounded_repeats = tensor(rounded_repeats, device='cuda')
        follow_up_transitions_torch = repeat_interleave(follow_up_transitions_torch, rounded_repeats)
        follow_up_transitions = cp.asarray(follow_up_transitions_torch)
        
        # no np.insert in cupy or pytorch
        insert_at = transition_indices + 1
        transition_series_new = cp.full(shape=(transition_series.size + insert_at.size), fill_value=-1)
        sorted_indices = cp.argsort(insert_at)
        add_to_indices = cp.arange(0, insert_at.size)
        insert_at[sorted_indices] += add_to_indices
        transition_series_new[insert_at] = follow_up_transitions
        transition_series_new[cp.where(transition_series_new == -1)[0]] = transition_series
        transition_series = transition_series_new

    
    time_step_series = cp.empty(transition_series.size + 1, dtype=cp.float64)
    time_step_series[0] = 0
    for i, transition_time_distribution in enumerate(prediction.transition_time_distributions):
        indices = cp.where(transition_series == transitions.transition_df.index[i])[0]
        drawn_lifetimes = transition_time_distribution.rvs(indices.size)
        time_step_series[indices + 1] = drawn_lifetimes
    
    time_series = cp.empty_like(time_step_series, dtype=cp.float64)
    time_series[:] = cp.cumsum(time_step_series, dtype=cp.float64)

    return time_series, transition_series

In [12]:
x, y = fill_individual_transitions(prediction, transition_set, size=1e4, seed=3)